In [1]:
from pathlib import Path
import sys

# Find the repo root whether VS Code starts here or inside notebooks/.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

sys.path.insert(0, str(root / "src"))

import gradio as gr
from wm.demo import apply_attack, decode, embed, load_model, text_to_bits

checkpoint = root / "results/checkin3/runs/hard_fgas_s42/best.pt"
model = load_model(checkpoint)
print("Watermark model loaded.")

/Users/tae/miniconda3/envs/cmpe258-demo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Watermark model loaded.


In [2]:
def display_image(tensor):
    pixels = tensor[0].permute(1, 2, 0).clamp(0, 1)
    return (pixels * 255).round().byte().cpu().numpy()


def run_demo(image, message, attack, scale):
    if image is None:
        raise gr.Error("Upload an image first.")

    cover, watermarked, psnr = embed(model, image, message)
    attacked = apply_attack(watermarked, attack, scale)
    _, decoded, errors = decode(model, attacked, text_to_bits(message))

    return (
        display_image(cover),
        display_image(watermarked),
        display_image(attacked),
        f"Decoded: {decoded!r} | Bit errors: {errors}/32 | PSNR: {psnr:.2f} dB",
    )


with gr.Blocks() as demo:
    image = gr.Image(type="pil", label="Upload an image")
    message = gr.Textbox(value="CMPE", label="Message (up to 4 ASCII characters)")
    attack = gr.Dropdown(["clean", "resize"], value="resize", label="Attack")
    scale = gr.Slider(0.1, 1.0, value=0.5, step=0.05, label="Resize scale")
    run = gr.Button("Embed and decode")

    with gr.Row():
        cover = gr.Image(label="Prepared original")
        watermarked = gr.Image(label="Watermarked")
        attacked = gr.Image(label="After attack")

    result = gr.Textbox(label="Decoded result")

    run.click(
        fn=run_demo,
        inputs=[image, message, attack, scale],
        outputs=[cover, watermarked, attacked, result],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.
